# **1. pyannote**

### 1. 화자 분리
화자 분리는 긴 녹음 파일을 듣고 언데 누가 말했는지를 시간 구간으로 나누는 작업

```
0.0 ~ 2.4초 "안녕하세요. 회의를 시작하겠습니다." SPEAKER_00
2.8 ~ 2.5초 "네, 자료를 준비했습니다." SPEAKER_01
5.8 ~ 7.2초 "첫 번쨰 안건부터 보겠습니다." SPEAKER_00
8.0 ~ 10.1초 "자료 너무 좋은 것 같습니다." SPEAKER_01
```

> SPEAKER_00은 실명이나 사용자 ID가 아님. 같은 녹음에서 같은 사람이라고 판단한 구간에 붙인 임시 라벨

- ASR: 음성 인식
- VAD: 음성 활동 탐지. 음성/비음성 구간
- 화자 분리: 화자별 시간 구간
- 화자 식별: 별도의 등록 음성/인증 체계 필요 -> 이 보이스는 등록된 XXX인가?

### 2. 컴퓨터가 사람의 목소리를 구별하는 방법
- 사람도 전화 통화에서 얼굴을 보지 않고 상대를 구별
- 말투, 발음 습관, 성대/발성 특성, 음색과 억양 등이 다르기 때문
- AI는 이런 특징을 학습한 신경망으로 음성을 숫자 벡터(화자 임베딩)로 바꿈

### 처리 순서
1. 오디오 읽기: wav를 샘플 값으로 읽음. 16kHz 모노라면 1초에 샘플 16,000개
2. 음성 구산 분석/분할: 짧은 시간 창마다 발화 여부와 여러 화자가 활동하는지 추정. 무음과 발화 경계가 중요
3. 화자 임베딩: 각 발화 구간의 목소리 특성을 고정 길이 숫자 벡터로 표현. 문장 뜻보다는 화자 구별에 유용한 음향적 특징을 학습
4. 화자 군집화: 비슷한 임베딩을 묶어서 동일한 화자로 판단. 서로 다른 구간이 머리 떨어져 있어도 같은 화자로 연결할 수 있음
5. 시간축 재구성: 군집 결과를 원래 발화 시간에 다시 매핑하고 SPEAKER_00 같은 라벨을 부여

### 겹쳐 말하는 상황
- 두 사람이 4.0~4.5초에 동시에 말하면 일반 화자 분리 결과는 두 화자의 구간이 겹칠 수 있음
- exclusive_speaker_diarization은 같은 시간에 하나의 화자만 배정하도록 만든 결과 -> ASR 전사 결과 -> 실제 동시 발화를 없앤 것처럼 복원

```
강의실에서 수강생 A와 B가 번갈아 발표하는 상황
- 음성 구간 탐지:"지금 누가 말하고 있음"
- 특징 추출: " 이 구간은 낮고 거친 음색, 다음 구간은 밝은 음색"
- 군집화: "첫 번쨰와 세 번쨰 발화가 비슷하므로 SPEAKER_00"
- 시간 정보 결합: A는 0~2초, B는 3~5초, A는 6~8초"

> 하지만 목소리 높낮이 만으로 판단하지는 않음. 한 사람이 소리를 높이거나 속삭여도 같은 화자일 수 있고, 다른 두 사람의 목소리가 비슷할 수도 있음. 따라서 임베딩은 여러 음향 특징을 함께 학습

> 짧은 대답, 웃음, 배경 음악이 있는 경우, 마이크 변경, 겹침, 멀리서 말하는 소리 등은 구분할 정보가 부족하거나 달라져서 오류가 생길 수 있음
```

hugging face 링크
<br>https://huggingface.co/pyannote/speaker-diarization-community-1
<br><br>
토큰 필요
<br>https://huggingface.co/settings/tokens
<br>
<br>
데이터셋
<br>https://github.com/joelcpy/pyannote/blob/main/meeting.wav?utm_source=chatgpt.com

In [1]:
import soundfile as sf
import librosa
import numpy as np
import os
import torch
import random
from pathlib import Path
from dotenv import load_dotenv
from pyannote.audio import Pipeline  # pip install pyannote.audio 

c:\VSGit\14_Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
AUDIO_PATH = Path("meeting.wav")
PREPARED_PATH = Path("meeting_16k_mono.wav")
TARGET_SR = 16000

wave, original_sr = sf.read(AUDIO_PATH, dtype="float32")
print("원본 샘플링 레이트: ", original_sr, "원본 shape: ", wave.shape)

원본 샘플링 레이트:  16000 원본 shape:  (480000,)


In [3]:
if wave.ndim == 2:
    wave = wave.mean(axis=1)

if original_sr != TARGET_SR:
    wave = librosa.resample(wave, orig_sr=original_sr, target_sr=TARGET_SR)

sf.write(PREPARED_PATH, wave, TARGET_SR)
print(f"변환 완료: {PREPARED_PATH} / {len(wave) / TARGET_SR:.2f}초")

변환 완료: meeting_16k_mono.wav / 30.00초


In [4]:
load_dotenv()

True

In [5]:
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN_ERROR")

In [6]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [7]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HF_TOKEN)
pipeline.to(DEVICE)

In [8]:
# 오디오 파일 읽기
wave, sr = sf.read(PREPARED_PATH, dtype="float32")

# NumPy → PyTorch Tensor
waveform = torch.from_numpy(wave)

# (샘플 수,) → (1, 샘플 수)
waveform = waveform.unsqueeze(0)

# 오디오 데이터 직접 전달
audio = {
    "waveform": waveform,
    "sample_rate": sr
}

# 화자 분리 실행
output = pipeline(audio)

for segment, speaker in output.speaker_diarization:
    print(f"{segment.start:6.2f} ~ {segment.end:6.2f} | {speaker}")

c:\VSGit\14_Agent\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:102: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1927.)
  std = sequences.std(dim=-1, correction=1)


  6.73 ~   6.75 | SPEAKER_00
  6.75 ~   7.03 | SPEAKER_01
  7.03 ~   7.19 | SPEAKER_00
  7.59 ~   7.61 | SPEAKER_00
  7.61 ~   8.32 | SPEAKER_01
  8.32 ~   9.92 | SPEAKER_00
  9.92 ~  10.98 | SPEAKER_01
 10.46 ~  14.75 | SPEAKER_00
 14.31 ~  17.88 | SPEAKER_01
 18.02 ~  21.51 | SPEAKER_00
 18.15 ~  18.44 | SPEAKER_01
 21.77 ~  28.50 | SPEAKER_01
 27.86 ~  29.97 | SPEAKER_00


### 일반 화자 분리 vs Exclusive 화자 분리
- 두 결과 모두 누가 언제 말했는지는 알려주지만, 여러 화자가 동시에 말하는 구간은 어떻게 표현하는지에 차이가 있음
- 일반 화자 분리: 실제 대화에서 여러 화자가 동시에 발화하는 구간을 표현할 수 있음
    - A: 0초부터 5초까지 발화, B: 3초부터 7초까지 발화 (3~5초 구간에서는 A와 B가 동시에 말하고 있음)
- Exclusive 화자 분리: 하나의 시간 구간에 최대 한 명의 화자만 배정하도록 만든 결과
    - A: 0초부터 5초까지 발화, B: 3초부터 7초까지 발화 -> 실제로는 3~5초 구간에는 두 사람이 동시에 말했지만 Exclusive 결과에서는 한 번의 한 명의 화자만 표현
        - {'start': 6.730343750000001, 'end': 6.74721875, 'speaker': 'SPEAKER_00'}, {'start': 6.74721875, 'end': 7.03409375, 'speaker': 'SPEAKER_01'} >> 중첩 발화 정보 손실


In [9]:
def to_rows(annotation):
    return [
        {"start": seg.start, "end": seg.end, "speaker": speaker} for seg, speaker in annotation
    ]

In [10]:
regular_rows = to_rows(output.speaker_diarization)
exclusive_rows = to_rows(output.exclusive_speaker_diarization)
print("일반 분리: ", regular_rows[:10])
print("Exclusive: ", exclusive_rows[:10])

일반 분리:  [{'start': 6.730343750000001, 'end': 6.74721875, 'speaker': 'SPEAKER_00'}, {'start': 6.74721875, 'end': 7.03409375, 'speaker': 'SPEAKER_01'}, {'start': 7.03409375, 'end': 7.185968750000001, 'speaker': 'SPEAKER_00'}, {'start': 7.59096875, 'end': 7.607843750000001, 'speaker': 'SPEAKER_00'}, {'start': 7.607843750000001, 'end': 8.316593750000003, 'speaker': 'SPEAKER_01'}, {'start': 8.316593750000003, 'end': 9.919718750000001, 'speaker': 'SPEAKER_00'}, {'start': 9.919718750000001, 'end': 10.98284375, 'speaker': 'SPEAKER_01'}, {'start': 10.45971875, 'end': 14.745968750000003, 'speaker': 'SPEAKER_00'}, {'start': 14.30721875, 'end': 17.88471875, 'speaker': 'SPEAKER_01'}, {'start': 18.01971875, 'end': 21.512843750000002, 'speaker': 'SPEAKER_00'}]
Exclusive:  [{'start': 6.730343750000001, 'end': 6.74721875, 'speaker': 'SPEAKER_00'}, {'start': 6.74721875, 'end': 7.03409375, 'speaker': 'SPEAKER_01'}, {'start': 7.03409375, 'end': 7.185968750000001, 'speaker': 'SPEAKER_00'}, {'start': 7.5909

# **2 참가자별 목소리 등록 회의록**

In [14]:
import sounddevice as sd    # pip install sounddevice
from pyannote.audio import Model, Inference
from qwen_asr import Qwen3ASRModel

SR = 16000
MIN_RMS = 0.008

In [12]:
# 사람의 목소리를 숫자 벡터를 변화하여 서로 같은 사람인지 비교할 수 있도록 해줌
embedding_model = Model.from_pretrained("pyannote/embedding", token=HF_TOKEN)  # pip install omegaconf
embedding_model.to(DEVICE)

# 불러온 임베딩 모델을 실제 음성 데이터에 적용하기 위한 추론 객체를 생성
# window="whole": 음성을 어떤 단위로 처리할 것인지 설정. whole은 입력으로 전달된 전체 음성 구간을 하나의 단위로 처리
embedder = Inference(embedding_model, window="whole")

ASR_MODEL_ID = "Qwen/Qwen3-ASR-1.7B"
asr_dtype = (torch.bfloat16 if torch. cuda.is_bf16_supported() else torch.float32) if torch.cuda.is_available() else torch.float32

# 음성에 포함된 말을 텍스트로 변환
asr_model = Qwen3ASRModel.from_pretrained(
    ASR_MODEL_ID, dtype=asr_dtype, device_map = "cuda" if torch.cuda.is_available() else "cpu",
    max_inference_batch_size=1, max_new_tokens=256
)

c:\VSGit\14_Agent\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:197: Redirecting import of pytorch_lightning.callbacks.early_stopping.EarlyStopping to lightning.pytorch.callbacks.early_stopping.EarlyStopping
c:\VSGit\14_Agent\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:197: Redirecting import of pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint to lightning.pytorch.callbacks.model_checkpoint.ModelCheckpoint
c:\VSGit\14_Agent\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\migration.py:208: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
Lightning automatically upgraded your loaded checkpoint from v1.2.7 to v2.6.6. To apply the u

### 1. 화자 등록

In [15]:
def record_voice(seconds=12):
    print(f"{seconds}초 녹음 중... 자연스럽게 말하세요.")
    recording = sd.rec(int(SR*seconds), samplerate=SR, channels=1, dtype="float32")
    sd.wait()
    return recording[:, 0].copy()

def embedding_of(wave):
    wave = np.asanyarray(wave, dtype=np.float32).reshape(-1)
    if len(wave) < SR:
        raise ValueError("화자 식별에는 최소 1초 이상의 음성이 필요합니다.")
    tensor = torch.from_numpy(wave).unsqueeze(0)

    with torch.inference_mode():
        vector = np.asarray(embedder({"waveform": tensor, "sample_rate": SR})).reshape(-1)
    norm = np.linalg.norm(vector)

    if norm < 1e-10:
        raise ValueError("유효한 화자 특징을 추출하지 못했습니다.")
    return vector / norm

registered = {}


def enroll(name, seconds=12):
    name = name.strip()
    if not name:
        raise ValueError("이름을 입력하세요")
    audio = record_voice(seconds)
    rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
    if rms < MIN_RMS:
        raise ValueError(f"녹음 음량이 너무 작습니다. 다시 등록하세요")
    registered[name] = embedding_of(audio)
    print(f"등록 완료: {name} (RMS={rms:.4f})")

In [16]:
enroll("김사과")

12초 녹음 중... 자연스럽게 말하세요.


ValueError: 녹음 음량이 너무 작습니다. 다시 등록하세요

In [17]:
enroll("오렌지")

12초 녹음 중... 자연스럽게 말하세요.


ValueError: 녹음 음량이 너무 작습니다. 다시 등록하세요

In [18]:
print("현재 등록된 참석자: ", list(registered))

현재 등록된 참석자:  []


### 출력 예시

[11:49:01] 김사과: 오늘 회의를 시작하겠습니다.
[11:49:11] 오렌지: 네, 첫 번째 안건을 말씀드리겠습니다.
[11:50:03] 김사과: 예산안을 다시 검토해보세요.